In [ ]:

Search Medium
Write

Harinder Singh Sudwal
Get unlimited access to all of Medium for less than $1/week.
Become a member


Optimizing Hyperparameters with Grid Search: A Hands-On Tutorial
Learn How to Leverage Grid Search to Optimize Hyperparameters
Okan Yenigün
DevOps.dev
Okan Yenigün

·
Following

Published in
DevOps.dev

·
7 min read
·
May 7
1







Photo by Sigmund on Unsplash
Congratulations! Your article is live on our publication. Do consider submitting more articles. Don’t forget to follow us on https://blog.devops.dev/ & on Twitter (https://twitter.com/devops_blog)Hyperparameters in machine learning models are parameters that are not learned from the training data but are set before training. These parameters affect the behavior of the model during training and can have a significant impact on the model’s performance and ability to generalize to new data.

Examples of hyperparameters include the learning rate, which controls the step size taken during optimization, the regularization strength, which controls the degree of penalization applied to the model’s parameters to prevent overfitting, the number of hidden layers in a neural network, the number of trees in a random forest, and the kernel function used in a support vector machine.

Grid search is a hyperparameter tuning technique used in machine learning to find the optimal values for the hyperparameters of a model. It involves defining a grid of hyperparameter values to search over, and then exhaustively evaluating each combination of values in the grid.


Grid Search. Source
I will use Laptop Prices Dataset from Kaggle to demonstrate how to use Grid Search in Python. You can get the data here.

To begin with, we need to preprocess the dataset and prepare it for model development. Since this is not the main focus of the article, it has been covered briefly.

import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("laptopPrice.csv")
# there are some duplicates in the dataset
df = df.drop_duplicates()
df.reset_index(drop=True,inplace=True)

# some replacements

df['ram_gb'] = df['ram_gb'].str.replace(' GB', '').astype(int)
df['ssd'] = df['ssd'].str.replace(' GB', '').astype(int)
df['hdd'] = df['hdd'].str.replace(' GB', '').astype(int)
df['graphic_card_gb'] = df['graphic_card_gb'].str.replace(' GB', '').astype(int)
df['os_bit'] = df['os_bit'].str.replace('-bit', '').astype(int)
df["rating"] = df["rating"].replace({"1 star": "1 stars"})
df['rating'] = df['rating'].str.replace(' stars', '').astype(int)

mapping = {'ThinNlight': 1, 'Casual': 2, 'Gaming': 3}
df["weight"] = df["weight"].replace(mapping)

mapping = {'No warranty': 0, '1 year':1, '2 years':2, '3 years':3}
df["warranty"] = df["warranty"].replace(mapping)

mapping = {'No': 0, 'Yes': 1}
df["Touchscreen"] = df["Touchscreen"].replace(mapping)

mapping = {'No': 0, 'Yes': 1}
df["msoffice"] = df["msoffice"].replace(mapping)

df.head(10)

First 10 rows. Image by the author.
num_features = [feature for feature in df.columns if df[feature].dtype != 'object' and feature != "Price"]
cat_features = [feature for feature in df.columns if df[feature].dtype == 'object']
print("Numerical features: ", num_features)
print("Categorical featues:", cat_features)

"""
Numerical features:  ['ram_gb', 'ssd', 'hdd', 'os_bit', 'graphic_card_gb', 'weight', 'warranty', 'Touchscreen', 'msoffice', 'rating', 'Number of Ratings', 'Number of Reviews']
Categorical featues: ['brand', 'processor_brand', 'processor_name', 'processor_gnrtn', 'ram_type', 'os']
"""
# feature transformation

cat_transformer = OneHotEncoder(handle_unknown='ignore')
num_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', cat_transformer, cat_features),
        ('num', num_transformer, num_features)
    ])

X = preprocessor.fit_transform(df)
y = df["Price"].values.reshape(-1,1)

print(f"X: {X.shape}, y: {y.shape}")
# X: (802, 51), y: (802, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, shuffle=True)
Alright, we have now preprocessed both the input and target data, and they are ready to be utilized in a model.

Let’s train our base model. I will use XGBoost with its default hyperparameters.

model = xgb.XGBRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse_train = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Train RMSE: {rmse_train}, Test RMSE: {rmse_test}")
# Train RMSE: 2079.484814769608, Test RMSE: 19711.16864024852
Our model appears to be highly overfitted. Therefore, we will now incorporate Grid Search to address this issue.

from sklearn.model_selection import GridSearchCV

Now, let’s take a closer look at GridSearchCV class. The parameters that it accepts are as follows:

estimator is the model that will be used for training.
param_grid specifies the hyperparameter space to search over. It should be a dictionary or a list of dictionaries, where each dictionary contains a set of hyperparameters to try.
scoring is the metric used to evaluate the performance of the model. It can take many different forms, including strings, callable functions, and dictionaries of multiple metrics. Classification metrics: accuracy, precision, recall, f1. Regression metrics: neg_mean_squared_error, r2. Clustering metrics: adjusted_rand_score, silhoutte_score. These were the most popular ones, visit here for the whole list.
n_jobs specifies the number of CPU cores to use for parallelizing the computation. A value of -1 indicates that all available cores should be used.
refit specifies whether to refit the best estimator on the entire dataset using the best hyperparameters found during the search. By default, refit is set to True, which means that after the grid search is completed, the GridSearchCV object will automatically refit the best estimator on the entire dataset using the best hyperparameters that were found.
cv specifies the cross-validation splitting strategy. It can be an integer value to specify the number of folds, or a cross-validation generator, which can be used to define more advanced cross-validation strategies.
verbose controls the verbosity of the output during the search.
pre_dispatch is used to control the number of jobs that are launched in parallel during the grid search. It takes an integer value, which specifies the maximum number of jobs that can be launched at any given time. For example, if pre_dispatch=2, then no more than 2 jobs will be launched in parallel at any given time.
error_score is used to specify what score should be assigned to a combination of hyperparameters if it fails to complete the fitting process. During the grid search process, the GridSearchCV algorithm trains and evaluates a model for each combination of hyperparameters. However, sometimes the model may fail to fit or score due to reasons such as insufficient memory or numerical instability. In such cases, the GridSearchCV algorithm needs to assign a score to the failed combination of hyperparameters so that it can continue with the search.
return_train_score specifies whether to include training scores in the output.
param_grid = {
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 500],
    'max_depth': [3, 5],
    'colsample_bytree': [0.5, 0.9],
    'gamma': [0, 0.1, 0.5],
    'reg_alpha': [0, 1, 10],
    'reg_lambda': [0, 1, 10],
}

xgb = xgb.XGBRegressor(random_state=1)
grid_search = GridSearchCV(xgb, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1, 
                           scoring="neg_root_mean_squared_error", )

grid_search.fit(X_train, y_train)
#Fitting 5 folds for each of 432 candidates, totaling 2160 fits
GridSearchCV is performing 5-fold cross-validation (i.e., splitting the data into 5 parts and training the model 5 times, each time using a different part as the validation set) for each of the 432 different combinations of hyperparameters in the search space. This results in a total of 2160 fits (i.e., training the model and evaluating its performance 2160 times).

We have now created our grid search object. Next, let’s explore the available attributes that we can use.

best_estimator_ returns the estimator that was chosen as the best among all the candidates based on the scoring metric specified.
best_score_ returns the mean cross-validated score achieved by the best estimator on the test data.
best_params_ returns a dictionary of the hyperparameters that produced the best result.
cv_results_ returns a dictionary that contains detailed information about the performance of each combination of hyperparameters, including the mean and standard deviation of the cross-validation scores, the time taken to fit and score each model, and the values of the hyperparameters for each model.
best_index_ returns the index of the best hyperparameter combination in the cv_results_ dictionary.
scorer_ represents the scoring function used to evaluate the performance of the models during the grid search.
n_splits_ represents the number of folds used in the cross-validation procedure during the grid search.
refit_time_ represents the time it took to refit the best estimator on the entire dataset.
multimetric_ indicates whether multiple evaluation metrics were used during the grid search.
classes_ returns the unique class labels in the target variable.
n_features_in_ returns the number of features in the input data.
feature_names_in_ is the names of features seen during fit.
print("Best estimator: ", grid_search.best_estimator_)

"""
Best estimator:  XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.5,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.1, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=5, max_leaves=0, min_child_weight=1,
             missing=nan, monotone_constraints='()', n_estimators=100, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=1, reg_alpha=0,
             reg_lambda=0, ...)
"""
print("Best score: ", grid_search.best_score_)
print("Best hyperparameters: ", grid_search.best_params_)

"""
Best score:  -23298.387344638286
Best hyperparameters:  {'colsample_bytree': 0.5, 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'reg_alpha': 0, 'reg_lambda': 0}
"""
results_df = pd.DataFrame(grid_search.cv_results_)
results_df.head()

CV results dataframe. Image by the author.
print("Best index: ", grid_search.best_index_)
print("Best scorer: ", grid_search.scorer_)
print("Best n splits: ", grid_search.n_splits_)
print("Best refit time: ", grid_search.refit_time_)
print("Best multi metric: ", grid_search.multimetric_)
print("Best n features: ", grid_search.n_features_in_)

"""
Best index:  54
Best scorer:  make_scorer(mean_squared_error, greater_is_better=False, squared=False)
Best n splits:  5
Best refit time:  0.055130958557128906
Best multi metric:  False
Best n features:  51
"""
We can use the best model now. We’ve achieved a slight improvement.

best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
rmse_train = np.sqrt(mean_squared_error(y_train, best_model.predict(X_train)))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Train RMSE: {rmse_train}, Test RMSE: {rmse_test}")

"""
Train RMSE: 6502.070891973686, Test RMSE: 17419.48195947506
"""
GridSearchCV evaluates all possible combinations of the specified hyperparameters, which can help identify the best set of hyperparameters for a given model. And, it ensures that the same hyperparameters are used across different runs of the model, which facilitates reproducibility.

Evaluating all possible combinations of hyperparameters can be computationally expensive, especially for larger datasets and more complex models. It can also be prone to overfitting if the search space is too large, which can lead to poor generalization performance on unseen data.

Read More
The Magic of XGBoost
Understand and Implement XGBoost in Python
python.plainenglish.io

Cross-Validation Techniques for Machine Learning: A Guide to Improve Model Performance
Understand Different Techniques and How to Use Them for Better Model Evaluation
medium.com

A Practical Catalog to Model Evaluation Metrics for Machine Learning
Master the Art of Evaluating Machine Learning Models: A Comprehensive Guide to Understanding and Using Classification…
python.plainenglish.io

Boltzmann Machines
Introduction to Boltzmann Machines
medium.com

jQuery User Interface Widgets
Utilizing jQuery UI Widgets
medium.com

Sources
sklearn.model_selection.GridSearchCV
Exhaustive search over specified parameter values for an estimator. Important members are fit, predict. GridSearchCV…
scikit-learn.org

https://www.kaggle.com/datasets/anubhavgoyal10/laptop-prices-dataset/code

https://community.alteryx.com/t5/Data-Science/Hyperparameter-Tuning-Black-Magic/ba-p/449289

https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter

Machine Learning
Python
Regression
Hyperparameter Tuning
Sklearn
1





Okan Yenigün
DevOps.dev
Written by Okan Yenigün
988 Followers
·
Writer for 
DevOps.dev

Top Writer for Django | Design Patterns | Image Processing | Regression | Time Series Forecasting. Writes about Python | Scala | Go | ML&DL |

Following

More from Okan Yenigün and DevOps.dev
A Catalog For Design Patterns in Python
Okan Yenigün
Okan Yenigün

in

Towards Dev

A Catalog For Design Patterns in Python
Exploring Design Patterns: Concepts, Analogies, Advantages, Disadvantages, and Real-Life Use Cases.
19 min read
·
Mar 16
135



How DevOps Engineers Can Utilize ChatGPT To Become 5x More Efficient
Neil Shah
Neil Shah

in

DevOps.dev

How DevOps Engineers Can Utilize ChatGPT To Become 5x More Efficient
Become a Pro from Day 1

·
4 min read
·
Jul 27
131



Docker Compose Tips & Tricks You Should Know
Tate Galbraith
Tate Galbraith

in

DevOps.dev

Docker Compose Tips & Tricks You Should Know
What would developers lives be like without Docker? We’d probably all be fumbling about installing dependencies on top of each other…

·
7 min read
·
Jul 13
128



Design Patterns in Python: State Pattern
Okan Yenigün
Okan Yenigün

in

Dev Genius

Design Patterns in Python: State Pattern
The State Design Pattern Explained and Implemented in Python
3 min read
·
Mar 14
51



See all from Okan Yenigün
See all from DevOps.dev
Recommended from Medium
Churn prediction using Ensemble Techniques-II
Vignesh Gopalakrishnan
Vignesh Gopalakrishnan

Churn prediction using Ensemble Techniques-II
Overview
12 min read
·
Jul 23
1



Multiclass classification using CatBoost and SHAP in Python
lochie links
lochie links

Multiclass classification using CatBoost and SHAP in Python
What is Catboost and SHAP
9 min read
·
Jul 12
21

1



Lists



Predictive Modeling w/ Python
20 stories
·
282 saves
Principal Component Analysis for ML
Time Series Analysis
deep learning cheatsheet for beginner
Practical Guides to Machine Learning
10 stories
·
294 saves



Coding & Development
11 stories
·
110 saves



Natural Language Processing
522 stories
·
141 saves
HTML REPORT
Prathamesh Gadekar
Prathamesh Gadekar

in

Level Up Coding

Python Libraries for Lazy Data Scientists
Do you feel lethargic today? Use these five libraries to boost your productivity.
7 min read
·
Apr 7
477

2



Using optuna with sklearn the right way — Part 1
Walter Sperat
Walter Sperat

Using optuna with sklearn the right way — Part 1
Using optuna with sklearn correctly can get ugly fast, let's try and make it simpler
11 min read
·
Jul 11
68

2



Decoding & Differentiating Distributions
Abdullah Babar
Abdullah Babar

Decoding & Differentiating Distributions
Distributions lie at the core of statistics and are very important whenever the discussion of Data arises. I have already discussed about…
10 min read
·
5 days ago
5



Categorical Data Encoding Techniques
Krishnakanth Naik Jarapala
Krishnakanth Naik Jarapala

in

AI Skunks

Categorical Data Encoding Techniques
Introduction:
7 min read
·
Mar 13
13



See more recommendations